In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer , AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.bfloat16
)


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto", 
    trust_remote_code = True     
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name , trust_remote_code = True 
)

In [3]:
lora_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules = ['q_proj' , 'v_proj'],
    lora_dropout = 0.05,
    bias = 'none',
    task_type = TaskType.CAUSAL_LM
)

In [4]:
model = get_peft_model(model , lora_config)

In [5]:
dataset = load_dataset('openai/gsm8k' , 'main' , split = 'train[:200]')

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [6]:
dataset

Dataset({
    features: ['question', 'answer'],
    num_rows: 200
})

In [7]:
dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

In [8]:
def tokenize(batch):
    # format the text for models expected input string format
    texts = [
        f"### Instruction:\n{instruction}\n### Response:\n{out}"
        for instruction , out in zip(batch['question'] , batch['answer'])
    ]
    
    tokens = tokenizer(
        texts, 
        padding = 'max_length',
        max_length = 256,
        truncation = True,
        return_tensors = 'pt'
    )
    
    tokens['labels'] = tokens['input_ids'].clone()
    return tokens

In [9]:
tokenized_dataset = dataset.map(
    tokenize,
    batched = True,
    remove_columns = dataset.column_names
)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [11]:
training_args = TrainingArguments(
    output_dir = './tinyllama-math-lora',
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    learning_rate = 1e-3,
    num_train_epochs = 50,
    fp16 = True,
    logging_steps = 20,
    save_strategy = 'epoch',
    report_to = 'none',
    remove_unused_columns = False,
    label_names = ['labels']
)

In [13]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm = False
)

In [15]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_dataset,
    processing_class = tokenizer,
    data_collator = data_collator
) 

In [16]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
20,1.269100
40,0.967500
60,0.842700
80,0.740500
100,0.606100
120,0.501200
140,0.408100
160,0.309300
180,0.256700
200,0.201500


TrainOutput(global_step=650, training_loss=0.22809997907051674, metrics={'train_runtime': 553.0343, 'train_samples_per_second': 18.082, 'train_steps_per_second': 1.175, 'total_flos': 1.590741172224e+16, 'train_loss': 0.22809997907051674, 'epoch': 50.0})

In [17]:
model.save_pretrained("./tinyllama-lora-tuned-adapter-math")
tokenizer.save_pretrained("./tinyllama-lora-tuned-adapter-math")

('./tinyllama-lora-tuned-adapter-math\\tokenizer_config.json',
 './tinyllama-lora-tuned-adapter-math\\special_tokens_map.json',
 './tinyllama-lora-tuned-adapter-math\\chat_template.jinja',
 './tinyllama-lora-tuned-adapter-math\\tokenizer.model',
 './tinyllama-lora-tuned-adapter-math\\added_tokens.json',
 './tinyllama-lora-tuned-adapter-math\\tokenizer.json')

In [18]:
adapter_path = "./tinyllama-lora-tuned-adapter-math/"

In [19]:
bnb_config

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name , quantization_config = bnb_config,
    device_map  = 'cuda', trust_remote_code = True
).eval()

tokenizer = AutoTokenizer.from_pretrained(
    model_name , trust_remote_code = True 
)

In [21]:
temp_model = AutoModelForCausalLM.from_pretrained(
    model_name , quantization_config = bnb_config,
    device_map  = 'cuda', trust_remote_code = True
)

In [23]:
from peft import PeftModel
tuned_model = PeftModel.from_pretrained(temp_model, adapter_path)
merged_model = tuned_model.merge_and_unload()
merged_model.eval()

c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\peft\tuners\lora\bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm(

In [24]:
eval_ds = load_dataset('openai/gsm8k' , 'main' , split = 'train[:200]')
eval_ds = eval_ds.map(
    tokenize,
    batched = True,
    remove_columns = ['question' , 'answer']
)

In [25]:
from torch.utils.data import DataLoader
eval_loader = DataLoader(
    eval_ds,
    batch_size = 8,
    collate_fn = data_collator
)

In [27]:
import math
@torch.no_grad()
def compute_perplexity(model):
    losses = []
    
    for batch in eval_loader:
        batch = {key : val.to('cuda') for key , val in batch.items()}
        loss = model(**batch).loss
        losses.append(loss.item())
        
    return math.exp(sum(losses) / len(losses))

In [28]:
print(f"Base model perplexity score: {compute_perplexity(base_model):.2f}")
print(f"Tuned model perplexity score: {compute_perplexity(tuned_model):.2f}")

Base model perplexity score: 5.88
Tuned model perplexity score: 1.05


In [29]:
def generate_response(
    model, tokenizer, text, 
    max_new_tokens = 128, 
    temperature = 0.7, 
    top_p = 0.9, 
    device='cuda'):
    
    model.eval()
    model.to(device)
    
    # Format input like instruction-response style
    prompt = f"### Instruction:\n{text}\n### Response:\n"
    
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    response = output_text.split("### Response:")[-1].strip()
    return response

In [33]:
text = "I sold total 20 mangoes and each one mango I bought at 18 taka each mango. How much I earn by selling them?"

In [34]:
print(
    f"Base model response: {generate_response(model=base_model, tokenizer=tokenizer, text=text)}"
)

print(
    f"Tuned model response: {generate_response(model=tuned_model, tokenizer=tokenizer, text=text)}"
)

Base model response: I have sold 20 mangoes at 18 taka each mango, so my total earnings from selling them is 2800 taka (20 x 18).
Tuned model response: I sold 20 mangoes for 20 mangoes * 18 taka = <<20*18=480>>480 taka
I earn 480 taka because each mango is worth 20 US$ and total mangoes are 20 mangoes * 20 mangoes = <<20*20=400>>400 US$
#### 400 US$ because total mangoes are worth 20 mangoes * 20 mangoes = <<20*
